# Transform + Initial Bulk Load Notebook

Transform raw bronze data from `s3://is459-crypto-raw-data/bronze/` using PySpark.


## 1. Setup — AWS credentials + SparkSession


In [1]:
import os, multiprocessing, sys
from dotenv import load_dotenv, find_dotenv

# ── 1. Load .env FIRST ────────────────────────────────────────────────────────
dotenv_path = find_dotenv()
print(f".env found at: {dotenv_path}")
load_dotenv(dotenv_path=dotenv_path, override=True)

AWS_KEY = os.environ["AWS_ACCESS_KEY_ID"]
AWS_SECRET = os.environ["AWS_SECRET_ACCESS_KEY"]
AWS_REGION = os.environ["AWS_REGION"]
BUCKET = os.environ["S3_BUCKET_RAW"]
BRONZE_S3 = f"s3a://{BUCKET}/bronze"

# Silver layer — falls back to the raw bucket under a /silver prefix
SILVER_BUCKET = os.environ.get("S3_BUCKET_SILVER", BUCKET)
SILVER_S3 = f"s3a://{SILVER_BUCKET}/silver/ohlcv"

# ClickHouse connection (Python client only — no JDBC)
CH_HOST = os.environ.get("CLICKHOUSE_HOST", "localhost")
CH_PORT = os.environ.get("CLICKHOUSE_PORT", "8123")
CH_DB = os.environ.get("CLICKHOUSE_DB", "default")
CH_USER = os.environ.get("CLICKHOUSE_USER", "default")
CH_PASSWORD = os.environ.get("CLICKHOUSE_PASSWORD", "")

# ── 2. Pin worker Python to the same interpreter as the driver ────────────────
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

NUM_CORES = multiprocessing.cpu_count()  # 11
PARALLELISM = NUM_CORES * 3  # 33
print(f"Python executable : {sys.executable}")
print(f"Logical CPUs      : {NUM_CORES}  →  parallelism = {PARALLELISM}")

# ── 3. JVM / packages — must be set BEFORE importing pyspark ─────────────────
# ClickHouse JDBC removed — ClickHouse reads Parquet from S3 directly (§8).
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 8g "
    "--packages "
    "org.apache.hadoop:hadoop-aws:3.4.1,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

# ── 4. Import pyspark AFTER setting env vars ──────────────────────────────────
import boto3
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# ── 5. SparkSession ───────────────────────────────────────────────────────────
spark = (
    SparkSession.builder.master("local[*]")
    .appName("transform-dev")
    # ── AWS credentials ───────────────────────────────────────────────────────
    .config("spark.hadoop.fs.s3a.access.key", AWS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET)
    .config("spark.hadoop.fs.s3a.endpoint", f"s3.{AWS_REGION}.amazonaws.com")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    # ── S3A connection pool ───────────────────────────────────────────────────
    .config("spark.hadoop.fs.s3a.connection.maximum", "200")
    .config("spark.hadoop.fs.s3a.threads.max", "200")
    # ── S3A read throughput ───────────────────────────────────────────────────
    .config("spark.hadoop.fs.s3a.readahead.range", "8388608")  # 8 MB  (was 64 KB)
    .config("spark.hadoop.fs.s3a.block.size", "134217728")  # 128 MB (was 32 MB)
    .config("spark.hadoop.fs.s3a.input.fadvise", "sequential")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "bytebuffer")
    .config("spark.hadoop.fs.s3a.multipart.size", "67108864")  # 64 MB
    # ── JVM / GC ──────────────────────────────────────────────────────────────
    .config("spark.driver.memory", "8g")
    .config("spark.driver.maxResultSize", "4g")
    .config(
        "spark.driver.extraJavaOptions",
        "-XX:+UseG1GC "
        "-XX:G1HeapRegionSize=16m "
        "-XX:InitiatingHeapOccupancyPercent=35 "
        "-XX:+UseStringDeduplication",
    )
    # ── Serialisation ─────────────────────────────────────────────────────────
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer.max", "512m")
    # ── Memory ────────────────────────────────────────────────────────────────
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.1")  # 90% of pool to execution
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "2g")
    # ── Parallelism ───────────────────────────────────────────────────────────
    .config("spark.default.parallelism", str(PARALLELISM))  # 33
    .config("spark.sql.shuffle.partitions", str(PARALLELISM))  # 33
    # ── Shuffle I/O ───────────────────────────────────────────────────────────
    .config("spark.shuffle.file.buffer", "1m")
    .config("spark.reducer.maxSizeInFlight", "96m")
    .config("spark.shuffle.localDisk.file.output.buffer", "5m")
    # ── SQL / AQE ─────────────────────────────────────────────────────────────
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "134217728")
    .config("spark.sql.files.maxPartitionBytes", "134217728")
    .config("spark.sql.files.openCostInBytes", "67108864")
    # ── Network timeouts ──────────────────────────────────────────────────────
    .config("spark.network.timeout", "600s")
    .config("spark.sql.broadcastTimeout", "600")
    # ── macOS fixes ───────────────────────────────────────────────────────────
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.executor.processTreeMetrics.enabled", "false")
    .getOrCreate()
)

hadoop_ver = spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
print(f"Spark             : {spark.version}")
print(f"Hadoop            : {hadoop_ver}")
print(f"Bucket (raw)      : {BUCKET}")
print(f"Bucket (silver)   : {SILVER_BUCKET}")
print(f"ClickHouse        : {CH_HOST}:{CH_PORT}/{CH_DB}")
print(f"Default parallel  : {spark.sparkContext.defaultParallelism}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Driver mem        : {spark.conf.get('spark.driver.memory')}")
print("SparkSession ready")

# ── 6. boto3 client (listing only) ────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    region_name=AWS_REGION,
)


def list_s3_folders(prefix):
    """Return immediate sub-folder prefixes under a given prefix."""
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix, Delimiter="/")
    return [cp["Prefix"] for cp in resp.get("CommonPrefixes", [])]

.env found at: /Users/lunlun/Downloads/Github/IS459_Crypto_BigData_Pipeline/.env
Python executable : /usr/local/bin/python
Logical CPUs      : 11  →  parallelism = 33


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/28 09:26:52 WARN Utils: Your hostname, Chengs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.91.162.177 instead (on interface en0)
26/03/28 09:26:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/lunlun/.ivy2.5.2/cache
The jars for the packages stored in: /Users/lunlun/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7995b93a-efbf-4e9b-ae21-ae92f1252537;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.1 in central
	found software.amazon.awssdk#bundle;2.24.6 in central
	found org.wildfl

Spark             : 4.0.0
Hadoop            : 3.4.1
Bucket (raw)      : is459-crypto-raw-data
Bucket (silver)   : is459-crypto-raw-data
ClickHouse        : localhost:8123/default
Default parallel  : 33
Shuffle partitions: 33
Driver mem        : 8g
SparkSession ready


## 2. Ticker Discovery — `kaggle/` vs `binance/`


In [2]:
# boto3 list_objects — metadata only, no data read
kaggle_tickers = {
    p.rstrip("/").split("/")[-1] for p in list_s3_folders("bronze/kaggle/btc-price-1m/")
}
binance_tickers = {
    # folders are named symbol=AAVEUSDT/ — strip the "symbol=" prefix
    p.rstrip("/").split("symbol=")[-1]
    for p in list_s3_folders("bronze/binance2/")
}

in_both = sorted(kaggle_tickers & binance_tickers)
only_kaggle = sorted(kaggle_tickers - binance_tickers)
only_binance = sorted(binance_tickers - kaggle_tickers)

print(f"Kaggle  : {len(kaggle_tickers):>4}  tickers")
print(f"Binance : {len(binance_tickers):>4}  tickers")

print(f"\n── In both ({len(in_both)}) ─────────────────────────────────────────────")
for t in in_both:
    print(f"  {t}")

print(f"\n── Kaggle only ({len(only_kaggle)}) — not in Binance ─────────────────────")
for t in only_kaggle:
    print(f"  {t}")

print(f"\n── Binance only ({len(only_binance)}) — not in Kaggle ────────────────────")
for t in only_binance:
    print(f"  {t}")

Kaggle  :   50  tickers
Binance :   48  tickers

── In both (48) ─────────────────────────────────────────────
  AAVEUSDT
  ADAUSDT
  ALGOUSDT
  ATOMUSDT
  AVAXUSDT
  BATUSDT
  BCHUSDT
  BNBUSDT
  BTCUSDT
  CAKEUSDT
  CHZUSDT
  CRVUSDT
  DASHUSDT
  DEXEUSDT
  DOGEUSDT
  DOTUSDT
  ENAUSDT
  ENJUSDT
  ETCUSDT
  ETHUSDT
  FILUSDT
  GRTUSDT
  HBARUSDT
  ICPUSDT
  IOSTUSDT
  IOTAUSDT
  LINKUSDT
  LTCUSDT
  MANAUSDT
  NEARUSDT
  NEOUSDT
  QTUMUSDT
  RVNUSDT
  SANDUSDT
  SHIBUSDT
  SOLUSDT
  SUSHIUSDT
  TFUELUSDT
  THETAUSDT
  TIAUSDT
  TRXUSDT
  UNIUSDT
  VETUSDT
  XLMUSDT
  XRPUSDT
  XTZUSDT
  ZECUSDT
  ZILUSDT

── Kaggle only (2) — not in Binance ─────────────────────
  EOSUSDT
  STMXUSDT

── Binance only (0) — not in Kaggle ────────────────────


## 3. `kaggle/` Preprocessing

`symbol` column is derived from the S3 folder name.


### 3.1 Standardizing `kaggle` dataframe


In [3]:
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    DoubleType,
    StringType,
    TimestampType,
)

KAGGLE_SCHEMA = StructType(
    [
        StructField("timestamp", TimestampType(), True),
        StructField("open", DoubleType(), True),
        StructField("high", DoubleType(), True),
        StructField("low", DoubleType(), True),
        StructField("close", DoubleType(), True),
        StructField("volume", DoubleType(), True),
        StructField("close_time", TimestampType(), True),
        StructField("quote_asset_volume", DoubleType(), True),
        StructField("number_of_trades", LongType(), True),
        StructField("taker_buy_base_asset_volume", DoubleType(), True),
        StructField("taker_buy_quote_asset_volume", DoubleType(), True),
        StructField("ignore", StringType(), True),
    ]
)

ORDERED_COLS = [
    "timestamp",
    "symbol",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "close_time",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_asset_volume",
    "taker_buy_quote_asset_volume",
]


In [4]:
def clean_df(df):
    """Trim symbol, drop nulls, drop duplicate (timestamp, symbol) rows."""
    return (
        df.withColumn("symbol", F.trim(F.col("symbol")))
        .dropna()
        .dropDuplicates(["timestamp", "symbol"])
    )

### 3.2 Compiled Kaggle DataFrame

`kaggle_raw` — explicit schema, single-pass CSV read across all ticker folders.


In [5]:
kaggle_paths = [
    f"s3a://{BUCKET}/{p}" for p in list_s3_folders("bronze/kaggle/btc-price-1m/")
]

KAGGLE_EXCLUDE = {"EOSUSDT", "STMXUSDT"}

kaggle_raw = (
    spark.read.schema(KAGGLE_SCHEMA)
    .csv(kaggle_paths, header=True)
    .drop("ignore")
    .withColumn(
        "symbol", F.regexp_extract(F.input_file_name(), r"/btc-price-1m/([^/]+)/", 1)
    )
    .filter(~F.col("symbol").isin(KAGGLE_EXCLUDE))
    .select(ORDERED_COLS)
)

26/03/28 09:26:58 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [6]:
kaggle_clean = clean_df(kaggle_raw)

In [7]:
kaggle_raw.show(5, truncate=False)

+-------------------+--------+------+------+------+------+--------+-----------------------+------------------+----------------+---------------------------+----------------------------+
|timestamp          |symbol  |open  |high  |low   |close |volume  |close_time             |quote_asset_volume|number_of_trades|taker_buy_base_asset_volume|taker_buy_quote_asset_volume|
+-------------------+--------+------+------+------+------+--------+-----------------------+------------------+----------------+---------------------------+----------------------------+
|2020-10-15 03:00:00|AAVEUSDT|51.43 |59.0  |50.666|55.607|1369.481|2020-10-15 03:00:59.999|75607.779013      |361             |898.434                    |50012.70048                 |
|2020-10-15 03:01:00|AAVEUSDT|55.603|55.623|52.0  |52.845|1120.278|2020-10-15 03:01:59.999|59696.689968      |178             |658.949                    |35330.288555                |
|2020-10-15 03:02:00|AAVEUSDT|52.845|52.846|51.283|51.999|466.672 |2020-10-

## 4. `binance/` Preprocessing

### 4.1 Compiled Binance DataFrame

`binance_raw` — explicit schema skips JSON schema-inference (avoids a second full S3 scan). Ticker folders resolved via boto3; passed as an explicit list so S3A does not attempt wildcard expansion.


In [8]:
# binance2/ uses Hive-style partitioning: symbol=AAVEUSDT/
# basePath tells Spark the partition root so it auto-adds the "symbol" column
BINANCE2_BASE = f"s3a://{BUCKET}/bronze/binance2"

binance_paths = [f"s3a://{BUCKET}/{p}" for p in list_s3_folders("bronze/binance2/")]

binance_raw = (
    spark.read.option("basePath", BINANCE2_BASE)
    .parquet(*binance_paths)
    .select(
        F.col("timestamp"),
        F.col("symbol"),  # from partition path symbol=AAVEUSDT/
        F.col("open").cast(DoubleType()),
        F.col("high").cast(DoubleType()),
        F.col("low").cast(DoubleType()),
        F.col("close").cast(DoubleType()),
        F.col("volume").cast(DoubleType()),
        F.col("close_time"),
        F.col("quote_asset_volume").cast(DoubleType()),
        F.col("number_of_trades"),
        F.col("taker_buy_base_asset_volume").cast(DoubleType()),
        F.col("taker_buy_quote_asset_volume").cast(DoubleType()),
    )
    .select(ORDERED_COLS)
)

In [9]:
binance_clean = clean_df(binance_raw)

In [10]:
binance_raw.show(5, truncate=False)

ERROR:root:KeyboardInterrupt while sending command.                 (0 + 1) / 1]
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

## 5. Union — Kaggle ∪ Binance

Both DataFrames share `ORDERED_COLS` — `unionByName` is schema-safe and avoids positional mismatches. No cache needed: `combined_df` is the terminal result, materialised once by the downstream write.


In [11]:
combined_df = kaggle_clean.unionByName(binance_clean)

## 6. Data Quality Filters

Applied once on `combined_df` before materialisation:

- **Positive prices** — zero/negative OHLC values are corrupt rows
- **OHLC consistency** — `high` must be ≥ all other prices; `low` must be ≤ all others
- **Non-negative volume** — negative volume is impossible


In [12]:
combined_df = combined_df.filter(
    # ── Positive prices ───────────────────────────────────────────────────────
    (F.col("open") > 0)
    & (F.col("high") > 0)
    & (F.col("low") > 0)
    & (F.col("close") > 0)
    &
    # ── OHLC consistency ──────────────────────────────────────────────────────
    (F.col("high") >= F.col("low"))
    & (F.col("high") >= F.col("open"))
    & (F.col("high") >= F.col("close"))
    & (F.col("low") <= F.col("open"))
    & (F.col("low") <= F.col("close"))
    &
    # ── Non-negative volume ───────────────────────────────────────────────────
    (F.col("volume") >= 0)
)

## 7. Final DataFrame


In [13]:
SAMPLE_TICKER = "BTCUSDT"

kaggle_sample = (
    spark.read.schema(KAGGLE_SCHEMA)
    .csv(f"s3a://{BUCKET}/bronze/kaggle/btc-price-1m/{SAMPLE_TICKER}/", header=True)
    .drop("ignore")
    .withColumn("symbol", F.lit(SAMPLE_TICKER))
    .select(ORDERED_COLS)
)

binance_sample = (
    spark.read.option("basePath", BINANCE2_BASE)
    .parquet(f"s3a://{BUCKET}/bronze/binance2/symbol={SAMPLE_TICKER}/")
    .select(ORDERED_COLS)
)

final_df = kaggle_sample.unionByName(binance_sample).filter(
    (F.col("open") > 0) & (F.col("high") >= F.col("low")) & (F.col("volume") >= 0)
)

In [15]:
final_df.show(5, truncate=False)

+-------------------+-------+-------+-------+-------+-------+--------+-----------------------+------------------+----------------+---------------------------+----------------------------+
|timestamp          |symbol |open   |high   |low    |close  |volume  |close_time             |quote_asset_volume|number_of_trades|taker_buy_base_asset_volume|taker_buy_quote_asset_volume|
+-------------------+-------+-------+-------+-------+-------+--------+-----------------------+------------------+----------------+---------------------------+----------------------------+
|2017-08-17 04:00:00|BTCUSDT|4261.48|4261.48|4261.48|4261.48|1.775183|2017-08-17 04:00:59.999|7564.90685084     |3               |0.075183                   |320.39085084                |
|2017-08-17 04:01:00|BTCUSDT|4261.48|4261.48|4261.48|4261.48|0.0     |2017-08-17 04:01:59.999|0.0               |0               |0.0                        |0.0                         |
|2017-08-17 04:02:00|BTCUSDT|4280.56|4280.56|4280.56|4280.56

## 7. Write — `cleaned/bq2_daily_prices/`


In [14]:
# DAILY_OUT = f"s3a://{BUCKET}/cleaned/bq2_daily_prices_initial_full_load"

# (
#     final_df
#     .withColumn("date", F.to_date("timestamp"))
#     .write
#     .mode("overwrite")
#     .option("compression", "snappy")
#     .partitionBy("date")
#     .parquet(DAILY_OUT)
# )

# print(f"Written to {DAILY_OUT}")